# Pancancer enrichment analysis step 2: Find enriched pathways

In [1]:
import cptac
import cptac.utils as ut
import pandas as pd
import numpy as np
import gseapy as gp
import os

In [2]:
# Read in the file from step 1
data = pd.read_csv("step1/tumor_changes.tsv", sep='\t', index_col=0)

# Only keep genes with P values below the cutoff
data = data[data["reject_null"]]

In [3]:
# For each cancer, find enriched pathways for genes that changed (either up or down)   
all_enrichments = pd.DataFrame()

for cancer_type in data["cancer_type"].unique():
    cancer_subset = data[data["cancer_type"] == cancer_type]
    changed_proteins = cancer_subset["protein_str"].tolist()
    enriched_pathways = gp.enrichr(
        gene_list=changed_proteins,
        description="changed_in_tumor",
        gene_sets="Reactome_2016",
        outdir=None)
    cancer_enriched = enriched_pathways.res2d.assign(cancer_type=cancer_type)
    all_enrichments = all_enrichments.append(cancer_enriched)

In [4]:
# Make a column of just pathway IDs
all_enrichments = all_enrichments.assign(pathway_id=all_enrichments["Term"].str.rsplit(" ", n=1, expand=True)[1])

# Make a column of overlap proportions
overlaps = all_enrichments["Overlap"].str.split("/", expand=True).astype("int")
overlap_props = overlaps[0] / overlaps[1]
all_enrichments = all_enrichments.assign(overlap_prop=overlap_props)

In [5]:
# Make a table with a list of all pathways, and:
# - The number of cancer types for which that cancer type is enriched
# - The average of the adjusted p values for that pathway
# - The average overlap proportion for that pathway
enrichment_summary = all_enrichments[["pathway_id", "Term"]].drop_duplicates(keep="first")

times_enriched = enrichment_summary["pathway_id"].apply(
    lambda x: all_enrichments[all_enrichments["pathway_id"] == x].shape[0])

avg_p_vals = enrichment_summary["pathway_id"].apply(
    lambda x: all_enrichments.loc[all_enrichments["pathway_id"] == x, "Adjusted P-value"].mean())

avg_overlap_props = enrichment_summary["pathway_id"].apply(
    lambda x: all_enrichments.loc[all_enrichments["pathway_id"] == x, "overlap_prop"].mean())

enrichment_summary = enrichment_summary.assign(
    times_enriched=times_enriched,
    avg_adj_p=avg_p_vals,
    avg_overlap_prop=avg_overlap_props)

enrichment_summary = enrichment_summary.sort_values(by=["times_enriched", "avg_adj_p"], ascending=[False, True])
enrichment_summary = enrichment_summary.reset_index(drop=True)

In [12]:
# Visualize selected pathways (this will show which genes are up or down)
# Make all increases +1 and decreases -1

to_viz = enrichment_summary["pathway_id"][0:1]

for pathway in to_viz:
    for cancer_type in data["cancer_type"].unique():
        

pandas.core.series.Series

In [7]:
enrichment_summary

,pathway_id,Term,times_enriched,avg_adj_p,avg_overlap_prop
0,R-HSA-1430728,Metabolism Homo sapiens R-HSA-1430728,8,1.247706e-37,0.571606
1,R-HSA-392499,Metabolism of proteins Homo sapiens R-HSA-392499,8,2.240574e-37,0.611383
2,R-HSA-74160,Gene Expression Homo sapiens R-HSA-74160,8,1.645517e-24,0.550276
3,R-HSA-72312,rRNA processing Homo sapiens R-HSA-72312,8,4.791297e-22,0.829167
4,R-HSA-199991,Membrane Trafficking Homo sapiens R-HSA-199991,8,6.457849e-21,0.644940
5,R-HSA-6791226,Major pathway of rRNA processing in the nucleo...,8,7.740814e-21,0.831325
6,R-HSA-5653656,Vesicle-mediated transport Homo sapiens R-HSA-...,8,1.372003e-18,0.611789
7,R-HSA-109582,Hemostasis Homo sapiens R-HSA-109582,8,4.313197e-18,0.613904
8,R-HSA-1643685,Disease Homo sapiens R-HSA-1643685,8,1.273541e-16,0.602241
9,R-HSA-5663205,Infectious disease Homo sapiens R-HSA-5663205,8,1.468345e-16,0.734195
